# 08 — Multi-Seed Robustness Check

Notebook 07 surfaced a validation/test ranking inversion (2D-only: best on val, worst on test) --
diagnosed as single-epoch early-stopping noise plus scaffold-split partition variance, both of
which are known small-N issues at hERG's scale (~600 molecules). This is exactly why TDC's own
leaderboard protocol averages over 5 scaffold-split seeds rather than reporting one.

This notebook re-trains the 5 core models (1D-only, 2D-only, 3D-only weighted_mean, multimodal
concat, multimodal gated) across **3 seeds**

The molecule universe (train_val union test) is identical across seeds -- only the train/valid
partition changes -- so the conformers cached in notebook 02 already cover every seed; nothing
needs to be regenerated.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGDataLoader
from tdc.benchmark_group import admet_group

from src.utils import load_json, save_json, set_seed
from src.featurizers import smiles_to_selfies, build_vocab, PAD
from src.datasets import (
    SelfiesDataset, ConformerDataset, collate_conformers, MultimodalDataset, multimodal_collate,
)
from src.graph_featurizer import mol_to_graph_data, ATOM_FEAT_DIM, BOND_FEAT_DIM
from src.models import Selfies1DModel, Graph2DModel, Conformer3DModel, MultimodalModel
from src.train_utils import train_binary_classifier, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [1, 2, 3, 4, 5]  

with open("../config/data.yaml") as f: cfg_data = yaml.safe_load(f)
with open("../config/model_1d.yaml") as f: cfg_1d = yaml.safe_load(f)
with open("../config/model_2d.yaml") as f: cfg_2d = yaml.safe_load(f)
with open("../config/model_3d.yaml") as f: cfg_3d = yaml.safe_load(f)
with open("../config/model_fusion.yaml") as f: cfg_fusion = yaml.safe_load(f)

group = admet_group(path=cfg_data["dataset"]["data_path"])
benchmark = group.get(cfg_data["dataset"]["name"])
test_df = benchmark["test"]

dropped_ids = {d["id"] for d in load_json("../data/processed/dropped_ids.json")}
print(f"Excluding {len(dropped_ids)} molecules that failed conformer generation, same as every earlier notebook.")

test_df = test_df[~test_df["Drug_ID"].isin(dropped_ids)].reset_index(drop=True)
print("Fixed test set size (same across all seeds):", len(test_df))


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Found local copy...


Excluding 4 molecules that failed conformer generation, same as every earlier notebook.
Fixed test set size (same across all seeds): 132


## Fixed test-set artifacts (built once, reused across all seeds)

In [2]:
vocab = load_json("../data/processed/selfies_vocab.json")  # same train-only vocab as every earlier notebook

test_smiles, test_ids, test_y = test_df["Drug"].tolist(), test_df["Drug_ID"].tolist(), test_df["Y"].tolist()
test_selfies = [smiles_to_selfies(s) for s in test_smiles]

test_selfies_ds = SelfiesDataset(test_selfies, test_y, vocab, max_len=cfg_1d["model"]["max_len"])
test_graph_ds = [mol_to_graph_data(smi, label) for smi, label in zip(test_smiles, test_y)]
test_conformer_ds = ConformerDataset(test_ids, test_y, cfg_3d["paths"]["conformers_dir"])
test_mm_ds = MultimodalDataset(
    ids=test_ids, smiles_list=test_smiles, selfies_list=test_selfies, labels=test_y,
    vocab=vocab, max_len=cfg_1d["model"]["max_len"], conformers_dir=cfg_3d["paths"]["conformers_dir"],
)

test_loaders = {
    "1d": DataLoader(test_selfies_ds, batch_size=32, shuffle=False),
    "2d": PyGDataLoader(test_graph_ds, batch_size=32, shuffle=False),
    "3d": DataLoader(test_conformer_ds, batch_size=32, shuffle=False, collate_fn=collate_conformers),
    "mm": DataLoader(test_mm_ds, batch_size=32, shuffle=False, collate_fn=multimodal_collate),
}
print("test loaders built")


test loaders built


## Model builders + forward_fns (same architectures as notebooks 03/04/05/06)

In [3]:
def build_1d():
    return Selfies1DModel(vocab_size=len(vocab), pad_id=vocab[PAD], **cfg_1d["model"])

def fwd_1d(model, batch, device):
    return model(batch["input_ids"].to(device))

def build_2d():
    return Graph2DModel(atom_feat_dim=ATOM_FEAT_DIM, bond_feat_dim=BOND_FEAT_DIM, **cfg_2d["model"])

def fwd_2d(model, batch, device):
    batch = batch.to(device)
    return model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

def build_3d():
    return Conformer3DModel(**cfg_3d["model"])  # agg_mode: learned_attention, locked in from notebook 05

def fwd_3d(model, batch, device):
    return model(
        batch["atom_z"].to(device), batch["atom_pos"].to(device),
        batch["atom_conf_batch"].to(device), batch["conf_mol_batch"].to(device),
        batch["conf_weights"].to(device), num_mols=batch["num_mols"],
    )

def build_mm(fusion_type):
    return MultimodalModel(
        vocab_size=len(vocab), pad_id=vocab[PAD],
        selfies_kwargs=dict(cfg_fusion["encoders"]["selfies"]),
        graph_kwargs={"atom_feat_dim": ATOM_FEAT_DIM, "bond_feat_dim": BOND_FEAT_DIM, **cfg_fusion["encoders"]["graph"]},
        schnet_kwargs=dict(cfg_fusion["encoders"]["schnet"]),
        fusion_type=fusion_type, fusion_dim=cfg_fusion["fusion"]["fusion_dim"],
    )

def fwd_mm(model, batch, device):
    graph = batch["graph"].to(device)
    return model(
        batch["input_ids"].to(device),
        graph.x, graph.edge_index, graph.edge_attr, graph.batch,
        batch["conf_atom_z"].to(device), batch["conf_atom_pos"].to(device),
        batch["conf_atom_conf_batch"].to(device), batch["conf_conf_mol_batch"].to(device),
        batch["conf_conf_weights"].to(device), batch["conf_num_mols"],
    )


## Per-seed training loop

For each seed: re-split train_val (test stays fixed), filter dropped ids, train all 5 models, evaluate each on the fixed test set. Checkpoints saved under `experiments/multiseed/<model>/seed<k>/` for the reproducibility record.

In [4]:
def get_seed_split(seed):
    train, valid = group.get_train_valid_split(benchmark=cfg_data["dataset"]["name"], split_type="default", seed=seed)
    train = train[~train["Drug_ID"].isin(dropped_ids)].reset_index(drop=True)
    valid = valid[~valid["Drug_ID"].isin(dropped_ids)].reset_index(drop=True)
    return train, valid


def build_seed_loaders(train_df, valid_df):
    train_smiles, valid_smiles = train_df["Drug"].tolist(), valid_df["Drug"].tolist()
    train_ids, valid_ids = train_df["Drug_ID"].tolist(), valid_df["Drug_ID"].tolist()
    train_y, valid_y = train_df["Y"].tolist(), valid_df["Y"].tolist()
    train_selfies = [smiles_to_selfies(s) for s in train_smiles]
    valid_selfies = [smiles_to_selfies(s) for s in valid_smiles]

    loaders = {}
    loaders["1d"] = {
        "train": DataLoader(SelfiesDataset(train_selfies, train_y, vocab, cfg_1d["model"]["max_len"]), batch_size=cfg_1d["train"]["batch_size"], shuffle=True),
        "valid": DataLoader(SelfiesDataset(valid_selfies, valid_y, vocab, cfg_1d["model"]["max_len"]), batch_size=cfg_1d["train"]["batch_size"], shuffle=False),
    }
    loaders["2d"] = {
        "train": PyGDataLoader([mol_to_graph_data(s, y) for s, y in zip(train_smiles, train_y)], batch_size=cfg_2d["train"]["batch_size"], shuffle=True),
        "valid": PyGDataLoader([mol_to_graph_data(s, y) for s, y in zip(valid_smiles, valid_y)], batch_size=cfg_2d["train"]["batch_size"], shuffle=False),
    }
    loaders["3d"] = {
        "train": DataLoader(ConformerDataset(train_ids, train_y, cfg_3d["paths"]["conformers_dir"]), batch_size=cfg_3d["train"]["batch_size"], shuffle=True, collate_fn=collate_conformers),
        "valid": DataLoader(ConformerDataset(valid_ids, valid_y, cfg_3d["paths"]["conformers_dir"]), batch_size=cfg_3d["train"]["batch_size"], shuffle=False, collate_fn=collate_conformers),
    }
    loaders["mm"] = {
        "train": DataLoader(MultimodalDataset(train_ids, train_smiles, train_selfies, train_y, vocab, cfg_1d["model"]["max_len"], cfg_3d["paths"]["conformers_dir"]), batch_size=cfg_fusion["train"]["batch_size"], shuffle=True, collate_fn=multimodal_collate),
        "valid": DataLoader(MultimodalDataset(valid_ids, valid_smiles, valid_selfies, valid_y, vocab, cfg_1d["model"]["max_len"], cfg_3d["paths"]["conformers_dir"]), batch_size=cfg_fusion["train"]["batch_size"], shuffle=False, collate_fn=multimodal_collate),
    }
    return loaders


In [5]:
all_results = []
model_specs = [
    ("1d_only",            build_1d,               fwd_1d, "1d", cfg_1d),
    ("2d_only",             build_2d,               fwd_2d, "2d", cfg_2d),
    ("3d_learned_attention",    build_3d,               fwd_3d, "3d", cfg_3d),
    ("multimodal_concat",  lambda: build_mm("concat"), fwd_mm, "mm", cfg_fusion),
    ("multimodal_gated",   lambda: build_mm("gated"),  fwd_mm, "mm", cfg_fusion),
]

for seed in SEEDS:
    print(f"\n=== SEED {seed} ===")
    set_seed(seed)
    train_df, valid_df = get_seed_split(seed)
    print(f"train={len(train_df)}  valid={len(valid_df)}  test={len(test_df)}")
    seed_loaders = build_seed_loaders(train_df, valid_df)

    for model_name, builder, forward_fn, modality, cfg in model_specs:
        exp_dir = f"../experiments/multiseed/{model_name}/seed{seed}"
        model = builder()
        model, train_info = train_binary_classifier(
            model, seed_loaders[modality]["train"], seed_loaders[modality]["valid"],
            forward_fn, cfg, exp_dir, device,
        )
        test_metrics = evaluate(model, test_loaders[modality], forward_fn, device)
        print(f"  [{model_name}] val_auroc={train_info['best_val_auroc']:.4f}  test_auroc={test_metrics['auroc']:.4f}")
        all_results.append({
            "model": model_name, "seed": seed,
            "val_auroc": train_info["best_val_auroc"], "test_auroc": test_metrics["auroc"],
        })
        del model
        torch.cuda.empty_cache()
save_json(all_results, "../experiments/multiseed/all_seed_results.json")
print("\ndone")


generating training, validation splits...



=== SEED 1 ===


100%|██████████| 523/523 [00:00<00:00, 1927.32it/s]


train=453  valid=66  test=132
epoch   1  train_loss=0.6247  val_loss=0.5226  val_auroc=0.7963
epoch   2  train_loss=0.5217  val_loss=0.4403  val_auroc=0.8187
epoch   3  train_loss=0.5255  val_loss=0.4519  val_auroc=0.8425
epoch   4  train_loss=0.4794  val_loss=0.4535  val_auroc=0.8300
epoch   5  train_loss=0.4552  val_loss=0.4301  val_auroc=0.8363
epoch   6  train_loss=0.4034  val_loss=0.4144  val_auroc=0.8400
epoch   7  train_loss=0.3996  val_loss=0.4266  val_auroc=0.8213
epoch   8  train_loss=0.3462  val_loss=0.4427  val_auroc=0.8400
epoch   9  train_loss=0.3088  val_loss=0.4630  val_auroc=0.8250
epoch  10  train_loss=0.2738  val_loss=0.4609  val_auroc=0.8350
epoch  11  train_loss=0.2604  val_loss=0.6341  val_auroc=0.8325
epoch  12  train_loss=0.4561  val_loss=0.6919  val_auroc=0.7913
epoch  13  train_loss=0.3596  val_loss=0.4536  val_auroc=0.8113
Early stopping at epoch 13 (no val AUROC improvement for 10 epochs)
  [1d_only] val_auroc=0.8425  test_auroc=0.7162
epoch   1  train_loss=

generating training, validation splits...


...

train=453  valid=66  test=132
epoch   1  train_loss=0.6353  val_loss=0.5532  val_auroc=0.7599
epoch   2  train_loss=0.5325  val_loss=0.4965  val_auroc=0.7743
epoch   3  train_loss=0.4849  val_loss=0.4858  val_auroc=0.7707
epoch   4  train_loss=0.4571  val_loss=0.5744  val_auroc=0.7575
epoch   5  train_loss=0.4212  val_loss=0.4622  val_auroc=0.7635
epoch   6  train_loss=0.4088  val_loss=0.4688  val_auroc=0.7827
epoch   7  train_loss=0.3590  val_loss=0.6194  val_auroc=0.7947
epoch   8  train_loss=0.3576  val_loss=0.5463  val_auroc=0.7935
epoch   9  train_loss=0.3241  val_loss=0.6401  val_auroc=0.8007
epoch  10  train_loss=0.3745  val_loss=0.5701  val_auroc=0.8055
epoch  11  train_loss=0.3196  val_loss=0.5321  val_auroc=0.8043
epoch  12  train_loss=0.3220  val_loss=0.5148  val_auroc=0.8067
epoch  13  train_loss=0.3121  val_loss=0.4795  val_auroc=0.8151
epoch  14  train_loss=0.2882  val_loss=0.5027  val_auroc=0.8175
epoch  15  train_loss=0.2932  val_loss=0.5091  val_auroc=0.8343
epoch  16 

generating training, validation splits...



=== SEED 5 ===


100%|██████████| 523/523 [00:00<00:00, 1773.88it/s]


train=453  valid=66  test=132
epoch   1  train_loss=0.5852  val_loss=0.6586  val_auroc=0.8120
epoch   2  train_loss=0.4912  val_loss=0.7395  val_auroc=0.8443
epoch   3  train_loss=0.4726  val_loss=0.5718  val_auroc=0.8604
epoch   4  train_loss=0.4478  val_loss=0.6259  val_auroc=0.8594
epoch   5  train_loss=0.4156  val_loss=0.5352  val_auroc=0.8547
epoch   6  train_loss=0.4012  val_loss=0.5330  val_auroc=0.8623
epoch   7  train_loss=0.3371  val_loss=0.5046  val_auroc=0.8253
epoch   8  train_loss=0.3544  val_loss=0.5728  val_auroc=0.7968
epoch   9  train_loss=0.3547  val_loss=0.6008  val_auroc=0.7635
epoch  10  train_loss=0.2943  val_loss=0.5735  val_auroc=0.8034
epoch  11  train_loss=0.3354  val_loss=0.5990  val_auroc=0.7892
epoch  12  train_loss=0.2996  val_loss=0.5719  val_auroc=0.8148
epoch  13  train_loss=0.2799  val_loss=0.6330  val_auroc=0.7930
epoch  14  train_loss=0.2669  val_loss=0.6672  val_auroc=0.7768
epoch  15  train_loss=0.2482  val_loss=0.6212  val_auroc=0.8025
epoch  16 

## Aggregate: mean +/- std, same format as the official leaderboard

In [6]:
df = pd.DataFrame(all_results)
summary = df.groupby("model")["test_auroc"].agg(["mean", "std", "count"]).reset_index()
summary.columns = ["model", "test_auroc_mean", "test_auroc_std", "n_seeds"]
summary = summary.sort_values("test_auroc_mean", ascending=False).reset_index(drop=True)
summary["test_auroc_display"] = summary.apply(lambda r: f"{r['test_auroc_mean']:.3f} +/- {r['test_auroc_std']:.3f}", axis=1)
summary


,model,test_auroc_mean,test_auroc_std,n_seeds,test_auroc_display
0,3d_learned_attention,0.818085,0.032939,5,0.818 +/- 0.033
1,multimodal_concat,0.785155,0.054045,5,0.785 +/- 0.054
2,multimodal_gated,0.773579,0.036386,5,0.774 +/- 0.036
3,2d_only,0.750869,0.013558,5,0.751 +/- 0.014
4,1d_only,0.731811,0.043921,5,0.732 +/- 0.044


In [7]:
save_json(summary.to_dict(orient="records"), "../experiments/multiseed/summary_table.json")
print("Saved multi-seed summary table.")


Saved multi-seed summary table.
